# Build Lineage Dependency Graph

This notebook builds a lineage dependency graph from extracted Delta tables.

It focuses on two dependency views:
1. Report visual to semantic model dependencies (direct to columns/measures/tables)
2. Semantic model internal dependencies (table, column, measure)

## Inputs (created by extraction notebooks)
- `lineage_reports`
- `lineage_report_visuals`
- `lineage_report_semantic_model_objects`
- `lineage_semantic_models`
- `lineage_semantic_model_columns`
- `lineage_semantic_model_measures`
- `lineage_semantic_model_relationships`

## Outputs
- `lineage_graph_nodes` (type-agnostic with JSON attributes)
- `lineage_graph_nodes_enriched` (view with display names from source tables)
- `lineage_graph_edges` (type-agnostic with JSON attributes)
- `lineage_graph_lineage_paths` (optional summarized paths)

## Architecture
The graph uses a **type-agnostic storage** design:
- **Nodes table** stores: `node_id`, `entity_type`, `attributes` (JSON)
- **Edges table** stores: `edge_id`, `from_node_id`, `to_node_id`, `edge_type`, `attributes` (JSON)
- All entity-specific attributes (IDs, names, etc.) are stored in JSON format
- **Enriched view** parses JSON and JOINs with source tables to provide display names
- This design makes lookups uniform regardless of entity type

**Entity types:** report, visual, table, column, measure, lakehouse, warehouse, notebook

**Visual dependencies:** Visuals connect directly to columns, measures, or tables (no intermediate nodes)

The graph outputs are designed for two audiences:
- Developers: impact tracing before model/report changes
- Business users: easier requirement discussions using dependency context

## 1. Configuration and Imports

Import required libraries and define configuration settings for the graph building process.

In [ ]:
import json
import re
from datetime import datetime
from typing import Any, Dict, List, Optional, Set, Tuple

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

# Configuration
WRITE_MODE = "overwrite"  # overwrite for deterministic graph rebuild, or append for incremental
BUILD_PATH_SUMMARY = True

REQUIRED_INPUT_TABLES = [
    "lineage_reports",
    "lineage_report_visuals",
    "lineage_report_semantic_model_objects",
    "lineage_semantic_model_columns",
    "lineage_semantic_model_measures",
    "lineage_semantic_model_relationships",
    "workspace_artifacts",
   # "lineage_notebook_dependencies",
    #"lineage_lakehouses",
    "lineage_warehouses"
]

OUTPUT_TABLE_NODES = "lineage_graph_nodes"
OUTPUT_TABLE_EDGES = "lineage_graph_edges"
OUTPUT_TABLE_PATHS = "lineage_graph_lineage_paths"


print("Graph build configuration ready")
print(f"WRITE_MODE: {WRITE_MODE}")

StatementMeta(, b37da37f-b2a8-43d4-bc8b-e000a50df7dd, 5, Finished, Available, Finished, False)

Graph build configuration ready
WRITE_MODE: overwrite


## 2. Helper Functions

Define utility functions for:
- Delta table operations (check existence, load, write)
- Column mapping and name normalization
- Node ID generation
- DAX expression parsing for dependency extraction
- Generic node/edge creation with type-agnostic attributes

In [ ]:
def _delta_table_exists(table_name: str) -> bool:
    return bool(spark.catalog.tableExists(table_name))


def _load_delta_table(table_name: str) -> DataFrame:
    return spark.table(table_name)


def _drop_table_if_exists(table_name: str) -> None:
    if _delta_table_exists(table_name):
        spark.sql(f"DROP TABLE IF EXISTS `{table_name}`")


def _append_to_delta(table_name: str, rows: List[Dict], mode: str = "append") -> int:
    if not rows:
        return 0

    if mode == "overwrite":
        _drop_table_if_exists(table_name)

    field_names = sorted({key for row in rows for key in row.keys()})
    schema = StructType([StructField(field_name, StringType(), True) for field_name in field_names])
    normalized_rows = [
        tuple(None if row.get(field_name) is None else str(row.get(field_name)) for field_name in field_names)
        for row in rows
    ]

    df = spark.createDataFrame(normalized_rows, schema=schema)
    writer = df.write.format("delta").option("mergeSchema", "true")

    if mode == "overwrite":
        writer.mode("overwrite").saveAsTable(table_name)
    else:
        writer.mode("append").saveAsTable(table_name)

    return len(rows)


def _first_existing_column(df: DataFrame, candidates: List[str]) -> Optional[str]:
    cols = set(df.columns)
    for candidate in candidates:
        if candidate in cols:
            return candidate
    return None


def _coalesce_string(row: Dict, keys: List[str], default: str = "") -> str:
    for key in keys:
        value = row.get(key)
        if value is not None and str(value).strip() != "":
            return str(value)
    return default


def _normalize_name(value: str) -> str:
    cleaned = re.sub(r"[^a-zA-Z0-9_]+", "_", str(value).strip().lower())
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")
    return cleaned or "unknown"


def _make_node_id(entity_type: str, parts: List[str]) -> str:
    normalized_parts = [
        _normalize_name(part)
        for part in parts
        if part is not None and str(part).strip() != ""
    ]
    if not normalized_parts:
        normalized_parts = ["unknown"]
    return f"{entity_type}:{'|'.join(normalized_parts)}"


def _parse_dax_object_refs(expression: str) -> Set[Tuple[str, str]]:
    """
    Best-effort parser for references in DAX-like expressions.

    Returns tuples of (table_name, object_name) where:
    - table_name may be empty for measure-only refs like [Total Sales]
    - object_name is column/measure name
    """
    if not expression:
        return set()

    refs: Set[Tuple[str, str]] = set()

    pattern_table_column = re.compile(r"'([^']+)'\[([^\]]+)\]|([A-Za-z0-9_]+)\[([^\]]+)\]")
    for match in pattern_table_column.finditer(expression):
        quoted_table = match.group(1)
        quoted_obj = match.group(2)
        plain_table = match.group(3)
        plain_obj = match.group(4)

        table_name = quoted_table if quoted_table is not None else plain_table
        object_name = quoted_obj if quoted_obj is not None else plain_obj
        refs.add((str(table_name), str(object_name)))

    pattern_measure_only = re.compile(r"\[([^\]]+)\]")
    for match in pattern_measure_only.finditer(expression):
        measure_name = str(match.group(1))
        refs.add(("", measure_name))

    return refs


def _create_node(entity_type: str, node_id: str, **attributes) -> Dict[str, Any]:

    """print("Helper functions ready")

    Create a type-agnostic node with attributes stored as JSON.

    

    Args:    }

        entity_type: Type of entity (report, visual, table, column, etc.)        "attributes": json.dumps(clean_attrs) if clean_attrs else None,

        node_id: Unique node identifier        "edge_type": edge_type,

        **attributes: Key-value pairs of node attributes        "to_node_id": to_node_id,

            "from_node_id": from_node_id,

    Returns:        "edge_id": edge_id,

        Dict with node_id, entity_type, and attributes as JSON string    return {

    """    

    # Filter out None values    clean_attrs = {k: v for k, v in attributes.items() if v is not None}

    clean_attrs = {k: v for k, v in attributes.items() if v is not None}    # Filter out None values

        """

    return {        Dict with edge_id, from_node_id, to_node_id, edge_type, and attributes as JSON string

        "node_id": node_id,    Returns:

        "entity_type": entity_type,    

        "attributes": json.dumps(clean_attrs) if clean_attrs else None,        **attributes: Key-value pairs of edge attributes

    }        edge_type: Type of relationship

        to_node_id: Target node ID

        from_node_id: Source node ID

def _create_edge(edge_id: str, from_node_id: str, to_node_id: str, edge_type: str, **attributes) -> Dict[str, Any]:        edge_id: Unique edge identifier

    """    Args:

    Create a type-agnostic edge with attributes stored as JSON.    

StatementMeta(, b37da37f-b2a8-43d4-bc8b-e000a50df7dd, 6, Finished, Available, Finished, False)

Helper functions ready


## 3. Load and Validate Input Tables

Load all required input tables from the lakehouse and validate they exist before proceeding.

In [21]:
# Validate required inputs
missing_inputs = [t for t in REQUIRED_INPUT_TABLES if not _delta_table_exists(t)]
if missing_inputs:
    raise ValueError(f"Missing required input tables: {missing_inputs}")

print("All required input tables found")

# Load input tables
reports_df = _load_delta_table("lineage_reports")
visuals_df = _load_delta_table("lineage_report_visuals")
report_objects_df = _load_delta_table("lineage_report_semantic_model_objects")
model_columns_df = _load_delta_table("lineage_semantic_model_columns")
model_measures_df = _load_delta_table("lineage_semantic_model_measures")
model_relationships_df = _load_delta_table("lineage_semantic_model_relationships")

print("Input tables loaded successfully")

StatementMeta(, b37da37f-b2a8-43d4-bc8b-e000a50df7dd, 7, Finished, Available, Finished, False)

All required input tables found
Input tables loaded successfully


## 4. Extract Column Mappings

Map flexible column names from source tables to handle schema variations across different extraction runs.

In [22]:
# Extract flexible column mappings from DataFrames
report_id_col_reports = _first_existing_column(reports_df, ["report_id", "id", "object_id"])
report_name_col_reports = _first_existing_column(reports_df, ["display_name", "displayName", "report_name", "name", "title", "object_name"])
dataset_id_col_reports = _first_existing_column(reports_df, ["dataset_id", "semantic_model_id", "model_id"])

visual_name_col = _first_existing_column(visuals_df, ["display_name", "displayName", "visual_name", "visualName", "name", "title", "object_name", "visual_title"])
visual_id_col = _first_existing_column(visuals_df, ["visual_id", "visualId", "object_id", "id"])
report_id_col_visuals = _first_existing_column(visuals_df, ["report_id"])

report_id_col_objects = _first_existing_column(report_objects_df, ["report_id"])
report_name_col_objects = _first_existing_column(report_objects_df, ["report_name", "report_display_name", "report_title"])
dataset_id_col_objects = _first_existing_column(report_objects_df, ["dataset_id", "semantic_model_id", "model_id"])
object_name_col = _first_existing_column(report_objects_df, ["object_name", "name"])
object_table_col = _first_existing_column(report_objects_df, ["table_name", "table"])
object_type_col = _first_existing_column(report_objects_df, ["object_type", "type"])
visual_name_col_objects = _first_existing_column(report_objects_df, ["display_name", "displayName", "visual_name", "visualName", "visual", "name", "title", "object_name", "visual_title"])
visual_id_col_objects = _first_existing_column(report_objects_df, ["visual_id", "visualId", "object_id", "id"])

model_id_col_columns = _first_existing_column(model_columns_df, ["semantic_model_id", "dataset_id", "model_id"])
table_name_col_columns = _first_existing_column(model_columns_df, ["table_name", "table", "table_id"])
column_name_col_columns = _first_existing_column(model_columns_df, ["column_name", "name"])
column_expression_col = _first_existing_column(model_columns_df, ["expression", "formula", "dax_expression", "Measure Expression", "Column Expression"])
column_format_col = _first_existing_column(model_columns_df, ["format_string", "format", "formatString", "Format String"])
column_data_type_col = _first_existing_column(model_columns_df, ["data_type", "datatype", "dataType", "Data Type"])

model_id_col_measures = _first_existing_column(model_measures_df, ["semantic_model_id", "dataset_id", "model_id"])
table_name_col_measures = _first_existing_column(model_measures_df, ["table_name", "table", "table_id"])
measure_name_col = _first_existing_column(model_measures_df, ["measure_name", "name"])
measure_expression_col = _first_existing_column(model_measures_df, ["expression", "formula", "dax_expression", "Measure Expression"])
measure_format_col = _first_existing_column(model_measures_df, ["format_string", "format", "formatString", "Format String"])
measure_data_type_col = _first_existing_column(model_measures_df, ["data_type", "datatype", "dataType", "Data Type"])

model_id_col_rel = _first_existing_column(model_relationships_df, ["semantic_model_id", "dataset_id", "model_id"])
from_table_col = _first_existing_column(model_relationships_df, ["from_table", "table_from", "from_table_name"])
from_column_col = _first_existing_column(model_relationships_df, ["from_column", "column_from", "from_column_name"])
to_table_col = _first_existing_column(model_relationships_df, ["to_table", "table_to", "to_table_name"])
to_column_col = _first_existing_column(model_relationships_df, ["to_column", "column_to", "to_column_name"])

# Validate required columns
if not report_id_col_reports:
    raise ValueError("lineage_reports is missing required report id column")
if not all([report_id_col_objects, object_name_col]):
    raise ValueError("lineage_report_semantic_model_objects is missing required columns (report_id/object_name)")
if not all([table_name_col_columns, column_name_col_columns]):
    raise ValueError("lineage_semantic_model_columns is missing required columns (table_name/column_name)")
if not all([table_name_col_measures, measure_name_col]):
    raise ValueError("lineage_semantic_model_measures is missing required columns (table_name/measure_name)")

print("Column mappings extracted and validated")

StatementMeta(, 7d41e819-201b-4b34-a080-b5cd24c01924, 12, Finished, Available, Finished, False)

Column mappings extracted and validated


StatementMeta(, 0198be86-c3bd-4e12-8da9-d529fd68d2c2, 12, Finished, Available, Finished, False)

Column mappings extracted and validated


StatementMeta(, b37da37f-b2a8-43d4-bc8b-e000a50df7dd, 8, Finished, Available, Finished, False)

Column mappings extracted and validated


## 5. Build Report Lookup

Create a lookup dictionary mapping report IDs to their names and dataset IDs for efficient reference during graph construction.

In [ ]:
# Build report lookup and initialize graph data structures
report_lookup = {}
report_select_cols = [report_id_col_reports]
if report_name_col_reports:
    report_select_cols.append(report_name_col_reports)
if dataset_id_col_reports:
    report_select_cols.append(dataset_id_col_reports)

for row in reports_df.select(*report_select_cols).collect():
    row_dict = row.asDict(recursive=True)
    report_id = _coalesce_string(row_dict, [report_id_col_reports], default="")
    if not report_id:
        continue

    report_lookup[report_id] = {
        "report_name": _coalesce_string(
            row_dict,
            [report_name_col_reports or "", "display_name", "displayName", "report_name", "name", "title", "object_name"],
            default=report_id,
        ),
        "dataset_id": _coalesce_string(
            row_dict,
            [dataset_id_col_reports or "", "dataset_id", "semantic_model_id", "model_id"],
            default="",
        ),
    }

nodes: Dict[str, Dict] = {}
edges: List[Dict] = []

print(f"Built report lookup with {len(report_lookup)} reports")

StatementMeta(, 0198be86-c3bd-4e12-8da9-d529fd68d2c2, 13, Finished, Available, Finished, False)

Built report lookup with 1 reports


StatementMeta(, b37da37f-b2a8-43d4-bc8b-e000a50df7dd, 9, Finished, Available, Finished, False)

Built report lookup with 1 reports


## 6. Create Report Nodes

Generate graph nodes for each Power BI report in the lineage data.

In [ ]:
# Create report nodes (type-agnostic with JSON attributes)
for report_id, report_info in report_lookup.items():
    report_name = report_info["report_name"]
    dataset_id = report_info["dataset_id"]
    report_node_id = _make_node_id("report", [dataset_id, report_id, report_name])

    nodes[report_node_id] = _create_node(
        entity_type="report",
        node_id=report_node_id,
        dataset_id=dataset_id,
        report_id=report_id,
    )

print(f"Created {len(report_lookup)} report nodes")

StatementMeta(, b37da37f-b2a8-43d4-bc8b-e000a50df7dd, 10, Finished, Available, Finished, False)

Created 1 report nodes


## 7. Create Visual Nodes and Report→Visual Edges

Create nodes for report visuals and edges connecting reports to their visuals.

In [ ]:
# Create visual nodes and report->visual edges
visual_lookup_by_id = {}
visual_lookup_by_name = {}

if visual_name_col and report_id_col_visuals:
    visuals_rows = visuals_df.select(
        report_id_col_visuals,
        visual_name_col,
        *([visual_id_col] if visual_id_col else [])
    ).collect()

    for row in visuals_rows:
        row_dict = row.asDict(recursive=True)
        report_id = str(row_dict.get(report_id_col_visuals) or "")
        visual_name = str(row_dict.get(visual_name_col) or "unknown_visual")
        visual_id = str(row_dict.get(visual_id_col) or "") if visual_id_col else ""

        if not report_id:
            continue

        report_info = report_lookup.get(report_id, {})
        dataset_id = report_info.get("dataset_id", "")
        report_name = report_info.get("report_name", report_id)

        visual_node_id = _make_node_id("visual", [dataset_id, report_id, visual_id, visual_name])
        report_node_id = _make_node_id("report", [dataset_id, report_id, report_name])

        nodes[visual_node_id] = _create_node(
            entity_type="visual",
            node_id=visual_node_id,
            dataset_id=dataset_id,
            report_id=report_id,
            visual_id=visual_id or None,
        )

        if visual_id:
            visual_lookup_by_id[f"{report_id}|{visual_id}"] = visual_node_id
        visual_lookup_by_name[f"{report_id}|{_normalize_name(visual_name)}"] = visual_node_id

        edges.append(_create_edge(
            edge_id=_make_node_id("edge", [report_node_id, visual_node_id, "contains_visual"]),
            from_node_id=report_node_id,
            to_node_id=visual_node_id,
            edge_type="contains_visual",
            dataset_id=dataset_id,
            report_id=report_id,
        ))

print(f"Created {len(visual_lookup_by_id) + len(visual_lookup_by_name)} visual nodes and edges")

StatementMeta(, 0198be86-c3bd-4e12-8da9-d529fd68d2c2, 15, Finished, Available, Finished, False)

Created 11 visual nodes and edges


StatementMeta(, b37da37f-b2a8-43d4-bc8b-e000a50df7dd, 11, Finished, Available, Finished, False)

Created 11 visual nodes and edges


## 8. Process Report Visual Dependencies

Create edges connecting visuals directly to the tables, columns, and measures they use (no intermediate semantic_object nodes).

In [ ]:
# Process report visual dependencies (visual -> column/measure/table edges)
report_object_rows = report_objects_df.collect()
visual_dependency_count = 0

for row in report_object_rows:
    row_dict = row.asDict(recursive=True)

    report_id = str(row_dict.get(report_id_col_objects) or "")
    report_name = str(
        report_lookup.get(report_id, {}).get("report_name")
        or _coalesce_string(
            row_dict,
            [report_name_col_objects or "", "report_display_name", "display_name", "displayName", "report_name", "report_title"],
            default=report_id,
        )
        or report_id
    )
    dataset_id = str(
        report_lookup.get(report_id, {}).get("dataset_id")
        or _coalesce_string(
            row_dict,
            [dataset_id_col_objects or "", "dataset_id", "semantic_model_id", "model_id"],
            default="",
        )
    )

    visual_name = _coalesce_string(row_dict, [visual_name_col_objects or "", "display_name", "displayName", "visual_name", "visualName", "visual", "name", "title", "object_name", "visual_title"], default="unknown_visual")
    visual_id = _coalesce_string(row_dict, [visual_id_col_objects or "", "visual_id", "visualId", "object_id", "id"], default="")

    sm_object_name = _coalesce_string(row_dict, [object_name_col or "", "object_name", "name"], default="unknown_object")
    sm_table_name = _coalesce_string(row_dict, [object_table_col or "", "table_name", "table"], default="")
    sm_object_type = _coalesce_string(row_dict, [object_type_col or "", "object_type", "type"], default="unknown").lower()

    # Resolve visual node from lookup
    visual_node_id = None
    if visual_id:
        visual_node_id = visual_lookup_by_id.get(f"{report_id}|{visual_id}")
    if not visual_node_id:
        normalized_visual_name = _normalize_name(visual_name)
        visual_node_id = visual_lookup_by_name.get(f"{report_id}|{normalized_visual_name}")

    # Fallback: create visual node if not found in lookup
    if not visual_node_id:
        visual_node_id = _make_node_id("visual", [dataset_id, report_id, visual_id, visual_name])
        nodes[visual_node_id] = _create_node(
            entity_type="visual",
            node_id=visual_node_id,
            dataset_id=dataset_id,
            report_id=report_id,
            visual_id=visual_id or None,
        )
        report_node_id = _make_node_id("report", [dataset_id, report_id, report_name])
        edges.append(_create_edge(
            edge_id=_make_node_id("edge", [report_node_id, visual_node_id, "contains_visual"]),
            from_node_id=report_node_id,
            to_node_id=visual_node_id,
            edge_type="contains_visual",
            dataset_id=dataset_id,
            report_id=report_id,
        ))

    # Determine target entity type and create appropriate node ID
    # Map object_type to entity_type (column, measure, or table)
    if "column" in sm_object_type or "field" in sm_object_type:
        target_entity_type = "column"
        target_node_id = _make_node_id("column", [dataset_id, sm_table_name, sm_object_name])
        edge_type = "visual_uses_column"
    elif "measure" in sm_object_type:
        target_entity_type = "measure"
        target_node_id = _make_node_id("measure", [dataset_id, sm_table_name, sm_object_name])
        edge_type = "visual_uses_measure"
    elif "table" in sm_object_type:
        target_entity_type = "table"
        target_node_id = _make_node_id("table", [dataset_id, sm_table_name])
        edge_type = "visual_uses_table"
    else:
        # Default to column if type is unclear
        target_entity_type = "column"
        target_node_id = _make_node_id("column", [dataset_id, sm_table_name, sm_object_name])
        edge_type = "visual_uses_column"

    # Create edge from visual to column/measure/table (nodes will be created in sections 9-10)
    edges.append(_create_edge(
        edge_id=_make_node_id("edge", [visual_node_id, target_node_id, edge_type]),
        from_node_id=visual_node_id,
        to_node_id=target_node_id,

        edge_type=edge_type,print(f"Processed {visual_dependency_count} visual dependencies (direct to columns/measures/tables)")

        dataset_id=dataset_id,

        report_id=report_id,    visual_dependency_count += 1

    ))    

StatementMeta(, 7d41e819-201b-4b34-a080-b5cd24c01924, 16, Finished, Available, Finished, False)

Processed 96 semantic model object references


StatementMeta(, 0198be86-c3bd-4e12-8da9-d529fd68d2c2, 16, Finished, Available, Finished, False)

Processed 96 semantic model object references


StatementMeta(, b37da37f-b2a8-43d4-bc8b-e000a50df7dd, 12, Finished, Available, Finished, False)

IndentationError: unexpected indent (926987578.py, line 67)

## 9. Process Semantic Model Columns

Create nodes for tables and columns, and parse DAX expressions to identify column dependencies.

In [ ]:
# Process semantic model columns (internal dependencies)
column_nodes_by_table_name: Dict[Tuple[str, str, str], str] = {}
column_count = 0

for row in model_columns_df.collect():
    row_dict = row.asDict(recursive=True)

    dataset_id = str(row_dict.get(model_id_col_columns) or "")
    table_name = str(row_dict.get(table_name_col_columns) or "")
    column_name = str(row_dict.get(column_name_col_columns) or "")
    expression = str(row_dict.get(column_expression_col) or "")
    format_string = str(row_dict.get(column_format_col) or "") if column_format_col else ""
    data_type = str(row_dict.get(column_data_type_col) or "") if column_data_type_col else ""

    table_node_id = _make_node_id("table", [dataset_id, table_name])
    column_node_id = _make_node_id("column", [dataset_id, table_name, column_name])

    nodes[table_node_id] = _create_node(
        entity_type="table",
        node_id=table_node_id,
        dataset_id=dataset_id,
        table_name=table_name,
    )

    nodes[column_node_id] = _create_node(
        entity_type="column",
        node_id=column_node_id,
        dataset_id=dataset_id,
        table_name=table_name,
        column_name=column_name,
    )

    column_nodes_by_table_name[(dataset_id, table_name, column_name)] = column_node_id

    edges.append(_create_edge(
        edge_id=_make_node_id("edge", [table_node_id, column_node_id, "contains_column"]),
        from_node_id=table_node_id,
        to_node_id=column_node_id,
        edge_type="contains_column",
        dataset_id=dataset_id,
    ))

    # Parse DAX dependencies
    refs = _parse_dax_object_refs(expression)
    for ref_table, ref_object in refs:
        if ref_table and ref_object:
            source_col_node_id = _make_node_id("column", [dataset_id, ref_table, ref_object])
            edges.append(_create_edge(
                edge_id=_make_node_id("edge", [source_col_node_id, column_node_id, "column_depends_on"]),
                from_node_id=source_col_node_id,
                to_node_id=column_node_id,
                edge_type="column_depends_on",
                dataset_id=dataset_id,
                evidence=expression[:500],
            ))
    
    column_count += 1

print(f"Processed {column_count} semantic model columns")

StatementMeta(, 7d41e819-201b-4b34-a080-b5cd24c01924, 17, Finished, Available, Finished, False)

Processed 466 semantic model columns


StatementMeta(, 0198be86-c3bd-4e12-8da9-d529fd68d2c2, 17, Finished, Available, Finished, False)

Processed 466 semantic model columns


## 10. Process Semantic Model Measures

Create measure nodes and identify their dependencies on columns and other measures by parsing DAX expressions.

In [ ]:
# Process semantic model measures (internal dependencies)
measure_nodes_by_name: Dict[Tuple[str, str], str] = {}
measure_count = 0

for row in model_measures_df.collect():
    row_dict = row.asDict(recursive=True)

    dataset_id = str(row_dict.get(model_id_col_measures) or "")
    table_name = str(row_dict.get(table_name_col_measures) or "")
    measure_name = str(row_dict.get(measure_name_col) or "")
    expression = str(row_dict.get(measure_expression_col) or "")
    format_string = str(row_dict.get(measure_format_col) or "") if measure_format_col else ""
    data_type = str(row_dict.get(measure_data_type_col) or "") if measure_data_type_col else ""

    table_node_id = _make_node_id("table", [dataset_id, table_name])
    measure_node_id = _make_node_id("measure", [dataset_id, table_name, measure_name])

    nodes[table_node_id] = _create_node(
        entity_type="table",
        node_id=table_node_id,
        dataset_id=dataset_id,
        table_name=table_name,
    )

    nodes[measure_node_id] = _create_node(
        entity_type="measure",
        node_id=measure_node_id,
        dataset_id=dataset_id,
        table_name=table_name,
        measure_name=measure_name,
    )

    measure_nodes_by_name[(dataset_id, measure_name)] = measure_node_id

    edges.append(_create_edge(
        edge_id=_make_node_id("edge", [table_node_id, measure_node_id, "contains_measure"]),
        from_node_id=table_node_id,
        to_node_id=measure_node_id,
        edge_type="contains_measure",
        dataset_id=dataset_id,
    ))

    # Parse DAX dependencies
    refs = _parse_dax_object_refs(expression)
    for ref_table, ref_object in refs:
        if ref_table and ref_object:
            source_col_node_id = _make_node_id("column", [dataset_id, ref_table, ref_object])
            edges.append(_create_edge(
                edge_id=_make_node_id("edge", [source_col_node_id, measure_node_id, "measure_depends_on_column"]),
                from_node_id=source_col_node_id,
                to_node_id=measure_node_id,
                edge_type="measure_depends_on_column",
                dataset_id=dataset_id,
                evidence=expression[:500],
            ))
        elif ref_object:
            ref_measure_node = measure_nodes_by_name.get((dataset_id, ref_object))
            if ref_measure_node:
                edges.append(_create_edge(
                    edge_id=_make_node_id("edge", [ref_measure_node, measure_node_id, "measure_depends_on_measure"]),
                    from_node_id=ref_measure_node,
                    to_node_id=measure_node_id,
                    edge_type="measure_depends_on_measure",
                    dataset_id=dataset_id,
                    evidence=expression[:500],
                ))
    
    measure_count += 1

print(f"Processed {measure_count} semantic model measures")

StatementMeta(, 7d41e819-201b-4b34-a080-b5cd24c01924, 18, Finished, Available, Finished, False)

Processed 83 semantic model measures


StatementMeta(, 0198be86-c3bd-4e12-8da9-d529fd68d2c2, 18, Finished, Available, Finished, False)

Processed 83 semantic model measures


## 11. Process Semantic Model Relationships

Create edges representing relationships between tables in the semantic model.

In [ ]:
# Process semantic model relationships
relationship_count = 0

if all([from_table_col, from_column_col, to_table_col, to_column_col]):
    for row in model_relationships_df.collect():
        row_dict = row.asDict(recursive=True)
        dataset_id = str(row_dict.get(model_id_col_rel) or "")

        from_table = str(row_dict.get(from_table_col) or "")
        from_column = str(row_dict.get(from_column_col) or "")
        to_table = str(row_dict.get(to_table_col) or "")
        to_column = str(row_dict.get(to_column_col) or "")

        from_col_node_id = _make_node_id("column", [dataset_id, from_table, from_column])
        to_col_node_id = _make_node_id("column", [dataset_id, to_table, to_column])

        edges.append(_create_edge(
            edge_id=_make_node_id("edge", [from_col_node_id, to_col_node_id, "relationship"]),
            from_node_id=from_col_node_id,
            to_node_id=to_col_node_id,
            edge_type="relationship",
            dataset_id=dataset_id,
        ))
        
        relationship_count += 1

print(f"Processed {relationship_count} semantic model relationships")

StatementMeta(, 0198be86-c3bd-4e12-8da9-d529fd68d2c2, 19, Finished, Available, Finished, False)

Processed 0 semantic model relationships


## 12. Process Lakehouses

Create nodes for Fabric Lakehouses from the lineage_lakehouses table.

In [ ]:
# Process lakehouses
lakehouse_count = 0

if _delta_table_exists("lineage_lakehouses"):
    try:
        lakehouses_df = _load_delta_table("lineage_lakehouses")
        
        row_count = lakehouses_df.count()
        if row_count == 0:
            print("⚠ lineage_lakehouses table is empty - skipping lakehouse nodes")
        else:
            print(f"Processing {row_count} lakehouse(s)...")
            
            for row in lakehouses_df.collect():
                row_dict = row.asDict(recursive=True)
                
                lakehouse_id = str(row_dict.get("lakehouse_id", ""))
                lakehouse_name = str(row_dict.get("lakehouse_name", ""))
                workspace_id = str(row_dict.get("workspace_id", ""))
                display_name = str(row_dict.get("display_name", "") or lakehouse_name)
                description = str(row_dict.get("description", "") or "")
                count_tables = row_dict.get("count_tables", 0)
                count_files = row_dict.get("count_files", 0)
                
                # Create lakehouse node (type-agnostic with JSON attributes)
                lakehouse_node_id = _make_node_id("lakehouse", [workspace_id, lakehouse_id])
                
                nodes[lakehouse_node_id] = _create_node(
                    entity_type="lakehouse",
                    node_id=lakehouse_node_id,
                    lakehouse_id=lakehouse_id,
                    workspace_id=workspace_id,
                )
                
                lakehouse_count += 1
            
            print(f"✅ Processed {lakehouse_count} lakehouse node(s)")
            
    except Exception as e:
        print(f"⚠ Error reading lineage_lakehouses: {str(e)}")
        import traceback
        traceback.print_exc()
else:
    print("⚠ lineage_lakehouses table not found - skipping lakehouse nodes")

StatementMeta(, 7d41e819-201b-4b34-a080-b5cd24c01924, 20, Finished, Available, Finished, False)

⚠ lineage_lakehouses table not found - skipping lakehouse nodes


StatementMeta(, 0198be86-c3bd-4e12-8da9-d529fd68d2c2, 20, Finished, Available, Finished, False)

⚠ lineage_lakehouses table not found - skipping lakehouse nodes


## 13. Process Warehouses

Create nodes for Fabric Warehouses from the lineage_warehouses table.

In [ ]:
# Process warehouses
warehouse_count = 0

if _delta_table_exists("lineage_warehouses"):
    try:
        warehouses_df = _load_delta_table("lineage_warehouses")
        
        row_count = warehouses_df.count()
        if row_count == 0:
            print("⚠ lineage_warehouses table is empty - skipping warehouse nodes")
        else:
            print(f"Processing {row_count} warehouse(s)...")
            
            for row in warehouses_df.collect():
                row_dict = row.asDict(recursive=True)
                
                warehouse_id = str(row_dict.get("warehouse_id", ""))
                warehouse_name = str(row_dict.get("warehouse_name", ""))
                workspace_id = str(row_dict.get("workspace_id", ""))
                display_name = str(row_dict.get("display_name", "") or warehouse_name)
                description = str(row_dict.get("description", "") or "")
                count_tables = row_dict.get("count_tables", 0)
                
                # Create warehouse node (type-agnostic with JSON attributes)
                warehouse_node_id = _make_node_id("warehouse", [workspace_id, warehouse_id])
                
                nodes[warehouse_node_id] = _create_node(
                    entity_type="warehouse",
                    node_id=warehouse_node_id,
                    warehouse_id=warehouse_id,
                    workspace_id=workspace_id,
                )
                
                warehouse_count += 1
            
            print(f"✅ Processed {warehouse_count} warehouse node(s)")
            
    except Exception as e:
        print(f"⚠ Error reading lineage_warehouses: {str(e)}")
        import traceback
        traceback.print_exc()
else:
    print("⚠ lineage_warehouses table not found - skipping warehouse nodes")

StatementMeta(, 0198be86-c3bd-4e12-8da9-d529fd68d2c2, 21, Finished, Available, Finished, False)

Processing 1 warehouse(s)...
✅ Processed 1 warehouse node(s)


## 14. Process Workspace Artifacts

Create nodes for Fabric workspace artifacts including notebooks, lakehouses, and warehouses.

In [ ]:
# Process workspace artifacts (notebooks, lakehouses, warehouses)
artifact_count = 0

if _delta_table_exists("workspace_artifacts"):
    try:
        artifacts_df = _load_delta_table("workspace_artifacts")
        
        # Check if table has any data
        row_count = artifacts_df.count()
        if row_count == 0:
            print("⚠ workspace_artifacts table is empty - skipping artifact nodes")
        else:
            print(f"Processing {row_count} artifact rows...")
            
            for row in artifacts_df.collect():
                row_dict = row.asDict(recursive=True)
                
                artifact_type = str(row_dict.get("artifact_type", "")).lower()
                artifact_id = str(row_dict.get("artifact_id", ""))
                artifact_name = str(row_dict.get("artifact_name", ""))
                workspace_id = str(row_dict.get("workspace_id", ""))
                description = str(row_dict.get("description", "") or "")
                
                # Create node for artifact (type-agnostic with JSON attributes)
                artifact_node_id = _make_node_id(artifact_type, [workspace_id, artifact_id, artifact_name])
                
                nodes[artifact_node_id] = _create_node(
                    entity_type=artifact_type,
                    node_id=artifact_node_id,
                    artifact_id=artifact_id,
                    workspace_id=workspace_id,
                )
                
                artifact_count += 1
            
            print(f"Processed {artifact_count} workspace artifacts")
            
    except Exception as e:
        print(f"⚠ Error reading workspace_artifacts: {str(e)}")
        print("  Skipping artifact nodes")
else:
    print("⚠ workspace_artifacts table not found - skipping artifact nodes")

StatementMeta(, 0198be86-c3bd-4e12-8da9-d529fd68d2c2, 22, Finished, Available, Finished, False)

Processing 37 artifact rows...
Processed 37 workspace artifacts


## 15. Process Notebook Dependencies

Create edges representing notebook dependencies on lakehouses and warehouses.

In [ ]:
# Process notebook dependencies
dependency_count = 0

if _delta_table_exists("lineage_notebook_dependencies"):
    try:
        dependencies_df = _load_delta_table("lineage_notebook_dependencies")
        
        # Check if table has any data
        row_count = dependencies_df.count()
        if row_count == 0:
            print("⚠ lineage_notebook_dependencies table is empty - skipping dependency edges")
        else:
            print(f"Processing {row_count} dependency rows...")
            
            # Show schema for debugging
            print(f"Table schema: {dependencies_df.columns}")
            
            for row in dependencies_df.collect():
                row_dict = row.asDict(recursive=True)
                
                notebook_id = str(row_dict.get("notebook_id", ""))
                notebook_name = str(row_dict.get("notebook_name", ""))
                workspace_id = str(row_dict.get("workspace_id", ""))
                dependency_type = str(row_dict.get("dependency_type", "")).lower()
                dependency_name = str(row_dict.get("dependency_name", ""))
                dependency_id = row_dict.get("dependency_id")
                extraction_method = str(row_dict.get("extraction_method", ""))
                
                # Skip placeholder/error dependencies
                if dependency_type in ["unknown", "error"] or dependency_name in ["pending_analysis", ""]:
                    continue
                
                # Create notebook node ID
                notebook_node_id = _make_node_id("notebook", [workspace_id, notebook_id, notebook_name])
                
                # Create dependency node ID (lakehouse or warehouse)
                if dependency_id:
                    dependency_node_id = _make_node_id(dependency_type, [workspace_id, dependency_id, dependency_name])
                else:
                    dependency_node_id = _make_node_id(dependency_type, [workspace_id, dependency_name])
                
                # Create edge: notebook uses lakehouse/warehouse
                edge_type = f"notebook_uses_{dependency_type}"
                edges.append(_create_edge(
                    edge_id=_make_node_id("edge", [notebook_node_id, dependency_node_id, edge_type]),
                    from_node_id=notebook_node_id,
                    to_node_id=dependency_node_id,
                    edge_type=edge_type,
                    extraction_method=extraction_method or None,
                ))
                
                dependency_count += 1
            
            print(f"Processed {dependency_count} notebook dependencies")
            
    except Exception as e:
        print(f"⚠ Error reading lineage_notebook_dependencies: {str(e)}")
        print("  Skipping dependency edges")
        import traceback
        traceback.print_exc()
else:
    print("⚠ lineage_notebook_dependencies table not found - skipping dependency edges")

StatementMeta(, 0198be86-c3bd-4e12-8da9-d529fd68d2c2, 23, Finished, Available, Finished, False)

⚠ lineage_notebook_dependencies table not found - skipping dependency edges


## 16. Write Graph to Delta Tables

Deduplicate edges and write both nodes and edges to Delta tables. Also create a path summary table grouping edges by type.

In [ ]:
# Write nodes and edges to Delta tables
# Remove duplicate edges by edge_id
unique_edges: Dict[str, Dict] = {}
for edge in edges:
    unique_edges[edge["edge_id"]] = edge

edge_rows = list(unique_edges.values())
node_rows = list(nodes.values())

print(f"Built {len(node_rows)} graph node(s)")
print(f"Built {len(edge_rows)} graph edge(s)")

nodes_written = _append_to_delta(OUTPUT_TABLE_NODES, node_rows, mode=WRITE_MODE)
edges_written = _append_to_delta(OUTPUT_TABLE_EDGES, edge_rows, mode=WRITE_MODE)

print(f"Wrote {nodes_written} node row(s) to {OUTPUT_TABLE_NODES}")
print(f"Wrote {edges_written} edge row(s) to {OUTPUT_TABLE_EDGES}")

# Build path summary
if BUILD_PATH_SUMMARY:
    paths_df = (
        spark.table(OUTPUT_TABLE_EDGES)
        .groupBy("edge_type")
        .agg(F.count("*").alias("edge_count"))
        .orderBy("edge_type")
    )

    if WRITE_MODE == "overwrite":
        _drop_table_if_exists(OUTPUT_TABLE_PATHS)
    paths_df.write.format("delta").mode("overwrite").saveAsTable(OUTPUT_TABLE_PATHS)
    print(f"Wrote path summary table: {OUTPUT_TABLE_PATHS}")

StatementMeta(, 0198be86-c3bd-4e12-8da9-d529fd68d2c2, 24, Finished, Available, Finished, False)

Built 526 graph node(s)
Built 491 graph edge(s)
Wrote 526 node row(s) to lineage_graph_nodes
Wrote 491 edge row(s) to lineage_graph_edges
Wrote path summary table: lineage_graph_lineage_paths


## 17. Create Enriched Nodes View

Create a convenience view that parses JSON attributes and JOINs with source tables to provide display names and metadata on demand. This view makes querying easy while keeping storage type-agnostic.

In [ ]:
# Create enriched view with metadata from source tables (parsing JSON attributes)
OUTPUT_VIEW_NODES_ENRICHED = "lineage_graph_nodes_enriched"

enriched_view_sql = f"""
CREATE OR REPLACE VIEW {OUTPUT_VIEW_NODES_ENRICHED} AS

-- Report nodes
SELECT 
    n.node_id,
    n.entity_type,
    n.attributes,
    get_json_object(n.attributes, '$.dataset_id') AS dataset_id,
    get_json_object(n.attributes, '$.report_id') AS report_id,
    COALESCE(r.{report_name_col_reports}, r.display_name, r.displayName, r.name, get_json_object(n.attributes, '$.report_id')) AS display_name,
    'report' AS object_subtype
FROM {OUTPUT_TABLE_NODES} n
LEFT JOIN lineage_reports r ON get_json_object(n.attributes, '$.report_id') = r.{report_id_col_reports}
WHERE n.entity_type = 'report'

UNION ALL

-- Visual nodes
SELECT 
    n.node_id,
    n.entity_type,
    n.attributes,
    get_json_object(n.attributes, '$.dataset_id') AS dataset_id,
    get_json_object(n.attributes, '$.report_id') AS report_id,
    COALESCE(v.{visual_name_col}, v.display_name, v.displayName, v.name, 'visual') AS display_name,
    'visual' AS object_subtype
FROM {OUTPUT_TABLE_NODES} n
LEFT JOIN lineage_report_visuals v 
    ON get_json_object(n.attributes, '$.report_id') = v.{report_id_col_visuals} 
    AND (get_json_object(n.attributes, '$.visual_id') = v.{visual_id_col} OR get_json_object(n.attributes, '$.visual_id') IS NULL)
WHERE n.entity_type = 'visual'

UNION ALL

-- Table nodes
SELECT 
    n.node_id,
    n.entity_type,
    n.attributes,
    get_json_object(n.attributes, '$.dataset_id') AS dataset_id,
    NULL AS report_id,
    get_json_object(n.attributes, '$.table_name') AS display_name,
    'table' AS object_subtype
FROM {OUTPUT_TABLE_NODES} n
WHERE n.entity_type = 'table'

UNION ALL

-- Column nodes
SELECT 
    n.node_id,
    n.entity_type,
    n.attributes,
    get_json_object(n.attributes, '$.dataset_id') AS dataset_id,
    NULL AS report_id,
    CONCAT(get_json_object(n.attributes, '$.table_name'), '.', get_json_object(n.attributes, '$.column_name')) AS display_name,
    'column' AS object_subtype
FROM {OUTPUT_TABLE_NODES} n
WHERE n.entity_type = 'column'

UNION ALL

-- Measure nodes
SELECT 
    n.node_id,
    n.entity_type,
    n.attributes,
    get_json_object(n.attributes, '$.dataset_id') AS dataset_id,
    NULL AS report_id,
    CONCAT(get_json_object(n.attributes, '$.table_name'), '.', get_json_object(n.attributes, '$.measure_name')) AS display_name,
    'measure' AS object_subtype
FROM {OUTPUT_TABLE_NODES} n
WHERE n.entity_type = 'measure'

UNION ALL

-- Lakehouse nodes
SELECT 
    n.node_id,
    n.entity_type,
    n.attributes,
    NULL AS dataset_id,
    NULL AS report_id,
    COALESCE(l.display_name, l.lakehouse_name, get_json_object(n.attributes, '$.lakehouse_id')) AS display_name,
    'lakehouse' AS object_subtype
FROM {OUTPUT_TABLE_NODES} n
LEFT JOIN lineage_lakehouses l 
    ON get_json_object(n.attributes, '$.lakehouse_id') = l.lakehouse_id 
    AND get_json_object(n.attributes, '$.workspace_id') = l.workspace_id
WHERE n.entity_type = 'lakehouse'

UNION ALL

-- Warehouse nodes
SELECT 
    n.node_id,
    n.entity_type,
    n.attributes,
    NULL AS dataset_id,
    NULL AS report_id,
    COALESCE(w.display_name, w.warehouse_name, get_json_object(n.attributes, '$.warehouse_id')) AS display_name,
    'warehouse' AS object_subtype
FROM {OUTPUT_TABLE_NODES} n
LEFT JOIN lineage_warehouses w 
    ON get_json_object(n.attributes, '$.warehouse_id') = w.warehouse_id 
    AND get_json_object(n.attributes, '$.workspace_id') = w.workspace_id
WHERE n.entity_type = 'warehouse'

UNION ALL

-- Other workspace artifact nodes (notebooks, etc.)
SELECT 
    n.node_id,
    n.entity_type,
    n.attributes,
    NULL AS dataset_id,
    NULL AS report_id,
    COALESCE(a.artifact_name, get_json_object(n.attributes, '$.artifact_id')) AS display_name,
    n.entity_type AS object_subtype
FROM {OUTPUT_TABLE_NODES} n
LEFT JOIN workspace_artifacts a 
    ON get_json_object(n.attributes, '$.artifact_id') = a.artifact_id 
    AND get_json_object(n.attributes, '$.workspace_id') = a.workspace_id
WHERE n.entity_type NOT IN ('report', 'visual', 'table', 'column', 'measure', 'lakehouse', 'warehouse')
"""

spark.sql(enriched_view_sql)
print(f"✅ Created enriched view: {OUTPUT_VIEW_NODES_ENRICHED}")
print(f"   This view parses JSON attributes and provides display names from source tables")
print(f"   Use this view for querying with a type-agnostic interface")


## 18. Impact Tracing Utilities

Helper functions to trace downstream impact of changes to any node in the lineage graph.

These functions use the enriched view (`lineage_graph_nodes_enriched`) which provides display names from source tables.

**Usage:**
```python
# Find a specific node
sample_node_id = find_node_id("measure", dataset_id="<dataset_id>", table_name="Sales", object_name="Total Sales")

# Trace all impacted nodes up to 4 levels deep
display(_trace_impacted_nodes(sample_node_id, max_depth=4))
```

In [ ]:
# Impact tracing helper functions
def _trace_impacted_nodes(start_node_id: str, max_depth: int = 3) -> DataFrame:
    """
    Trace all nodes impacted by changes to a starting node.
    
    Args:
        start_node_id: The node ID to start tracing from
        max_depth: Maximum depth to traverse (default: 3)
    
    Returns:
        DataFrame with impacted nodes and their depth from start node
    """
    edges_df = spark.table(OUTPUT_TABLE_EDGES).select("from_node_id", "to_node_id", "edge_type")

    frontier_schema = StructType([
        StructField("node_id", StringType(), True),
        StructField("depth", StringType(), True),
    ])
    frontier = spark.createDataFrame([(str(start_node_id), "0")], schema=frontier_schema)
    visited = frontier

    for depth in range(1, max_depth + 1):
        next_hop = (
            frontier.alias("f")
            .join(edges_df.alias("e"), F.col("f.node_id") == F.col("e.from_node_id"), "inner")
            .select(F.col("e.to_node_id").alias("node_id"))
            .distinct()
            .withColumn("depth", F.lit(str(depth)))
        )

        next_hop = next_hop.join(visited.select("node_id"), on="node_id", how="left_anti")
        if next_hop.limit(1).count() == 0:
            break

        visited = visited.unionByName(next_hop)
        frontier = next_hop

    return (
        visited.alias("v")
        .join(spark.table(OUTPUT_VIEW_NODES_ENRICHED).alias("n"), F.col("v.node_id") == F.col("n.node_id"), "left")
        .select("v.depth", "n.node_id", "n.entity_type", "n.display_name", "n.dataset_id", "n.object_subtype")
        .orderBy("depth", "entity_type", "display_name")
    )


def find_node_id(entity_type: str, dataset_id: str, table_name: str = "", object_name: str = "") -> str:
    """
    Construct a node ID from entity metadata.
    
    Args:
        entity_type: Type of entity (e.g., "measure", "column", "table")
        dataset_id: Dataset/semantic model ID
        table_name: Optional table name
        object_name: Optional object name
    
    Returns:
        Constructed node ID
    """
    parts = [dataset_id]
    if table_name:
        parts.append(table_name)
    if object_name:
        parts.append(object_name)
    return _make_node_id(entity_type, parts)


print("Impact tracing helpers ready")
print("\nExample usage:")
print('# sample_node_id = find_node_id("measure", dataset_id="<dataset_id>", table_name="Sales", object_name="Total Sales")')
print('# display(_trace_impacted_nodes(sample_node_id, max_depth=4))')

StatementMeta(, 7d41e819-201b-4b34-a080-b5cd24c01924, 26, Finished, Available, Finished, False)

Impact tracing helpers ready

Example usage:
# sample_node_id = find_node_id("measure", dataset_id="<dataset_id>", table_name="Sales", object_name="Total Sales")
# display(_trace_impacted_nodes(sample_node_id, max_depth=4))


StatementMeta(, 0198be86-c3bd-4e12-8da9-d529fd68d2c2, 25, Finished, Available, Finished, False)

Impact tracing helpers ready

Example usage:
# sample_node_id = find_node_id("measure", dataset_id="<dataset_id>", table_name="Sales", object_name="Total Sales")
# display(_trace_impacted_nodes(sample_node_id, max_depth=4))


## 19. Type-Agnostic Query Examples

The new type-agnostic design makes querying uniform across all entity types. Here are examples of how to leverage this:

In [ ]:
# Example 1: Query all nodes of any type - uniform schema
print("=" * 60)
print("Example 1: All nodes have the same schema (node_id, entity_type, attributes)")
print("=" * 60)
all_nodes_df = spark.table(OUTPUT_TABLE_NODES)
display(all_nodes_df.select("node_id", "entity_type", "attributes").limit(10))

# Example 2: Filter by entity type without knowing specific columns
print("\n" + "=" * 60)
print("Example 2: Filter by entity type - no need to know column names")
print("=" * 60)
reports_only = spark.table(OUTPUT_TABLE_NODES).filter(F.col("entity_type") == "report")
display(reports_only.limit(5))

# Example 3: Parse JSON attributes for specific fields
print("\n" + "=" * 60)
print("Example 3: Extract specific attributes using JSON parsing")
print("=" * 60)
columns_with_ids = (
    spark.table(OUTPUT_TABLE_NODES)
    .filter(F.col("entity_type") == "column")
    .select(
        "node_id",
        "entity_type",
        F.get_json_object("attributes", "$.dataset_id").alias("dataset_id"),
        F.get_json_object("attributes", "$.table_name").alias("table_name"),
        F.get_json_object("attributes", "$.column_name").alias("column_name"),
    )
)
display(columns_with_ids.limit(5))

# Example 4: Query edges - all have same schema
print("\n" + "=" * 60)
print("Example 4: All edges have uniform schema (edge_id, from_node_id, to_node_id, edge_type, attributes)")
print("=" * 60)
all_edges_df = spark.table(OUTPUT_TABLE_EDGES)
display(all_edges_df.select("edge_id", "from_node_id", "to_node_id", "edge_type").limit(10))

# Example 5: Use enriched view for easy display names
print("\n" + "=" * 60)
print("Example 5: Use enriched view to get display names automatically")
print("=" * 60)
enriched_nodes = spark.table(OUTPUT_VIEW_NODES_ENRICHED)
display(enriched_nodes.select("node_id", "entity_type", "display_name", "object_subtype").limit(10))

# Example 6: Generic lookup by node_id (works for any entity type)
print("\n" + "=" * 60)
print("Example 6: Lookup ANY node by ID - same query for all types")
print("=" * 60)
sample_node_id = all_nodes_df.first()["node_id"]
node_detail = spark.table(OUTPUT_VIEW_NODES_ENRICHED).filter(F.col("node_id") == sample_node_id)
display(node_detail)

print("\n✅ Type-agnostic design benefits:")
print("   • Uniform schema across all entity types")
print("   • Easy to add new entity types without schema changes")
print("   • Consistent querying patterns")
print("   • Flexible attribute storage via JSON")
print("   • Enriched view provides display names when needed")